In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path

import HumSpectra.fluorescence as fl
import HumSpectra.ultraviolet as uv
import HumSpectra.utilits as ut

In [2]:
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from Data_processing.config_paths import INIT_DATA_2024, INIT_DATA_2025, STAT_PATH

In [5]:
uv_path = Path.joinpath(INIT_DATA_2025, "UV","Individual_spectra")
uv_spectra = ut.load_spectra_data(uv_path, uv.read_csv_uv, spectra_type="absorption")
uv_spectra

[]

In [7]:
uv_path

WindowsPath('c:/Users/Kirill/Desktop/Lab/KNP-analysis/Initial_data/2025/UV/Individual_spectra')

In [10]:
date_list = ["2024","2025"]
date_folder_list = [INIT_DATA_2024,INIT_DATA_2025]

for i, folder in enumerate(date_folder_list):
    
    fluo_path = Path.joinpath(folder, "Fluorescence")
    uv_path = Path.joinpath(folder, "UV","Individual_spectra")

    carbon_data = pd.read_excel(Path.joinpath(folder, 'C_org.xlsx'))
    geodata = pd.read_excel(Path.joinpath(folder, 'GeoData.xlsx'))
    
    fluo_spectra = ut.load_spectra_data(fluo_path, fl.read_fluo_3d)
    uv_spectra = ut.load_spectra_data(uv_path, uv.read_csv_uv, spectra_type="absorption")

    fl_data = ut.spectra_to_df(fluo_spectra, {
        "Asm_350": fl.asm_350, "Asm_280": fl.asm_280,
        "FIX": fl.fluo_index, "HIX": fl.humin_index
    })

    uv_data = ut.spectra_to_df(uv_spectra, {
        "E2E3": uv.e2_e3, "epsilon": uv.epsilon
    })

    MainData = (uv_data.merge(carbon_data, on="Sample")
                    .assign(Suva=lambda x: 100*x.epsilon / x.C_org)
                    .drop(columns=["epsilon"])
                    .merge(geodata[["Sample", "pH", "eH"]], on="Sample")
                    .merge(fl_data, on=["Sample", "Class", "Subclass"]))

    numeric_cols = MainData.select_dtypes(include=['float']).columns
    MainData[numeric_cols] = MainData[numeric_cols].map(lambda x: float(f"{x:.4g}"))
    
    MainData.set_index(["Class","Subclass","Sample"],inplace=True)
    break

In [12]:
uv_data

,Sample,Class,Subclass,E2E3,epsilon
0,ADOM-SL10-1-UV,ADOM,Storage 10,4.047395,0.119148
1,ADOM-SL10-2-UV,ADOM,Storage 10,4.187010,0.068611
2,ADOM-SL2-1-UV,ADOM,Storage 2,3.921451,0.176933
3,ADOM-SL2-10-UV,ADOM,Storage 2,3.615338,0.931994
4,ADOM-SL2-11-UV,ADOM,Storage 2,3.519771,1.004107
...,...,...,...,...,...
78,ADOM-SL7-8-UV,ADOM,Storage 7,1.926844,0.029684
79,ADOM-SL7-9-UV,ADOM,Storage 7,4.883135,0.162405
80,ADOM-SL8-1-UV,ADOM,Storage 8,3.134566,0.023954
81,ADOM-SL9-1-UV,ADOM,Storage 9,3.632929,0.115729
